<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l6.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L6 · Backtest SMA con costos
Cruce SMA(10,30) sobre 120 días: señal con datos pasados, 5 bps por trade, equity bruto vs neto.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c2_l6_precios.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/python/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
FAST, SLOW = 10, 30
import numpy as np

COSTO = 0.0005  # 5 bps por cambio de posicion
df["ret"] = np.log(df["precio"] / df["precio"].shift(1)).fillna(0)
df["sma_fast"] = df["precio"].rolling(FAST).mean()
df["sma_slow"] = df["precio"].rolling(SLOW).mean()
df["pos"] = (df["sma_fast"].shift(1) > df["sma_slow"].shift(1)).astype(int).fillna(0)
df["bruto"] = df["pos"].shift(1).fillna(0) * df["ret"]
df["costo"] = df["pos"].diff().abs().fillna(0) * COSTO
df["neto"] = df["bruto"] - df["costo"]
df["equity"] = np.exp(df["neto"].cumsum())
trades = int(df["pos"].diff().abs().sum())
print(f"trades: {trades}  costo total: {df['costo'].sum():.4f} ({df['costo'].sum()*10000:.0f} bps)")
print(f"ret bruto: {np.exp(df['bruto'].sum())-1:+.2%}  ret neto: {df['equity'].iloc[-1]-1:+.2%}")

## El costo se come el brillo
El equity bruto ignora la fricción; el neto resta 5 bps por cada cambio de posición. Más trades, más arrastre.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["equity"], color="#5eead4", label="neto (con costos)")
ax.plot(__import__("numpy").exp(df["bruto"].cumsum()), color="#f59e0b", linestyle="--", label="bruto (sin costos)")
ax.set_ylabel("equity")
ax.set_title("Bruto vs neto: el arrastre de los costos")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
r = df["neto"]
sharpe = r.mean() / r.std(ddof=0) * (252 ** 0.5)
pico = df["equity"].cummax()
dd = df["equity"] / pico - 1
print(f"Sharpe neto anualizado: {sharpe:.2f}")
print(f"MaxDD: {dd.min():.2%}")

In [ ]:
# Chequeos automáticos
assert len(df) == 120, "se esperan 120 días"
assert trades == 5, "nº de trades determinista"
assert df["costo"].sum() > 0, "operar cuesta: el costo total es positivo"
assert df["equity"].iloc[-1] < __import__("numpy").exp(df["bruto"].sum()), "el neto siempre queda bajo el bruto"
print("OK: backtest con costos verificado")